# MVA v3 · NB04C — AlphaGenome RNA-mechanism audit
**v1.0.0 · public variants only · orthogonal mechanism triage**

## Purpose

This notebook does **not** use AlphaGenome to infer BUBR1 protein stability or to replace the N1002K functional assay.

It asks a different question:

> Do either of the two nominated BUB1B variants produce a strong predicted **RNA-level / cis-regulatory**
> effect — especially altered splicing, splice-junction usage, transcript abundance, or polyadenylation?

Variants:
- `chr15:40209701 T>G` — `NM_001211.6:c.2210T>G`, p.Leu737Ter
- `chr15:40220612 T>G` — `NM_001211.6:c.3006T>G`, p.Asn1002Lys

These two variants are already part of the sanitized/public project evidence package.

## Why this analysis is orthogonal to NB04B

NB04B showed that AlphaMissense, PolyPhen, SIFT, ESM2 and BLOSUM62 do **not** distinguish experimentally
low-abundance from WT-like BUBR1 substitutions.

AlphaGenome is therefore not used as another vote for "destabilized protein".
Its value here is to detect an alternative **DNA → RNA** mechanism that the protein-centric predictors cannot test.

## Frozen interpretation rules before viewing AlphaGenome output

Primary modalities:
- `SPLICE_SITES`
- `SPLICE_SITE_USAGE`
- `SPLICE_JUNCTIONS`
- `RNA_SEQ`
- `POLYADENYLATION`

Secondary discovery modalities:
- `CAGE`
- `PROCAP`
- `ATAC`
- `DNASE`
- `CHIP_TF`
- `CHIP_HISTONE`

An output is flagged `EXTREME` if `abs(quantile_score) >= 0.995`.

### Final mechanism call

`RNA_LEVEL_POSITIVE`
- at least one BUB1B primary-modality row has `abs(quantile_score) >= 0.995`, **and**
- the signal is reproducible in at least two tracks/biosamples or two independent primary modalities.

`RNA_LEVEL_SINGLE_SIGNAL_HOLD`
- exactly one extreme BUB1B primary signal without independent support.

`RNA_LEVEL_NOT_SUPPORTED`
- no extreme BUB1B primary-modality signal.

These are **research triage labels only**. AlphaGenome output is not PS3 and is not independently stacked on top of PP3.

## Consequences

- `RNA_LEVEL_POSITIVE` → add targeted RT-PCR / junction assay / RNA-seq to Stage F0 before protein rescue screening.
- `RNA_LEVEL_NOT_SUPPORTED` → keep the main functional hypothesis at protein abundance/function level.
- Either way, N1002K protein stability remains experimentally unresolved until directly measured.

In [ ]:
import os
# Install current official AlphaGenome client.
import importlib.util, subprocess, sys, os, json, math, time
if importlib.util.find_spec("alphagenome") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "alphagenome==0.9.0"])

import numpy as np
import pandas as pd
from pathlib import Path

from alphagenome.data import genome
from alphagenome.models import dna_client, variant_scorers

# Colab secret handling: never print the API key.
API_KEY=None
try:
    from google.colab import userdata
    API_KEY=userdata.get("ALPHA_GENOME_API_KEY")
except Exception:
    API_KEY=os.environ.get("ALPHA_GENOME_API_KEY")

assert API_KEY, (
    "Add ALPHA_GENOME_API_KEY to Colab Secrets or environment. "
    "Obtain the key from the official AlphaGenome access page."
)

model=dna_client.create(
    api_key=API_KEY,
    address="dns:///gdmscience.googleapis.com:443",
)

# Project Drive output.
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    ROOT = Path(os.environ.get("MVA_ROOT", "/path/to/MVA_Hackathon_2026"))
    PUBLIC_OUT=ROOT/"03_PUBLIC_ARTIFACTS"
    DOC_OUT=ROOT/"04_DOCUMENTATION"
    PUBLIC_OUT.mkdir(parents=True,exist_ok=True)
    DOC_OUT.mkdir(parents=True,exist_ok=True)
except Exception:
    PUBLIC_OUT=Path(".")
    DOC_OUT=Path(".")

print("AlphaGenome client initialized. API key not displayed.")

## 1. Public variant definitions

In [ ]:
VARIANTS = {
    "L737star": genome.Variant(
        chromosome="chr15",
        position=40209701,   # 1-based VCF coordinate
        reference_bases="T",
        alternate_bases="G",
    ),
    "N1002K": genome.Variant(
        chromosome="chr15",
        position=40220612,
        reference_bases="T",
        alternate_bases="G",
    ),
}

for name,v in VARIANTS.items():
    print(name, v)

PRIMARY = [
    "SPLICE_SITES",
    "SPLICE_SITE_USAGE",
    "SPLICE_JUNCTIONS",
    "RNA_SEQ",
    "POLYADENYLATION",
]

SECONDARY = [
    "CAGE",
    "PROCAP",
    "ATAC",
    "DNASE",
    "CHIP_TF",
    "CHIP_HISTONE",
]

SCORERS = [
    variant_scorers.RECOMMENDED_VARIANT_SCORERS[x]
    for x in PRIMARY + SECONDARY
]

## 2. Score both variants over 1-Mb sequence context

In [ ]:
all_rows=[]

for variant_name,variant in VARIANTS.items():
    interval=variant.reference_interval.resize(dna_client.SEQUENCE_LENGTH_1MB)

    print(f"Scoring {variant_name}: {variant}")
    scores=model.score_variant(
        interval=interval,
        variant=variant,
        variant_scorers=SCORERS,
    )

    tidy=variant_scorers.tidy_scores(
        scores,
        match_gene_strand=True,
        include_extended_metadata=True,
    )
    assert tidy is not None and len(tidy)>0

    tidy=tidy.copy()
    tidy["variant_name"]=variant_name
    all_rows.append(tidy)

scores_df=pd.concat(all_rows,ignore_index=True)

scores_df.to_csv(
    PUBLIC_OUT/"NB04C_ALPHAGENOME_ALL_PUBLIC_SCORES.csv",
    index=False,
)

print("Total AlphaGenome score rows:",len(scores_df))

## 3. Restrict primary interpretation to BUB1B

In [ ]:
# Gene-centric outputs should identify BUB1B directly.
# Non-gene-centric secondary modalities are kept separately and never create the primary mechanism call.

primary_df=scores_df[scores_df["output_type"].isin(PRIMARY)].copy()

if "gene_name" in primary_df.columns:
    bub_primary=primary_df[
        primary_df["gene_name"].fillna("").str.upper().eq("BUB1B")
    ].copy()
else:
    bub_primary=pd.DataFrame(columns=primary_df.columns)

assert len(bub_primary)>0, (
    "No gene-centric BUB1B rows found. Inspect AlphaGenome GENCODE mapping before interpretation."
)

bub_primary["abs_quantile"]=bub_primary["quantile_score"].abs()
bub_primary["extreme"]=bub_primary["abs_quantile"]>=0.995

cols=[
    c for c in [
        "variant_name","variant_id","gene_name","output_type","variant_scorer",
        "biosample_name","gtex_tissue","ontology_curie","track_name",
        "raw_score","quantile_score","abs_quantile","extreme"
    ] if c in bub_primary.columns
]

bub_primary[cols].sort_values(
    ["variant_name","abs_quantile"],
    ascending=[True,False],
).to_csv(
    PUBLIC_OUT/"NB04C_ALPHAGENOME_BUB1B_PRIMARY.csv",
    index=False,
)

display(
    bub_primary[cols].sort_values(
        ["variant_name","abs_quantile"],ascending=[True,False]
    ).head(80)
)

## 4. Frozen primary mechanism call

In [ ]:
def classify_variant(df):
    z=df[df.extreme].copy()
    if len(z)==0:
        return "RNA_LEVEL_NOT_SUPPORTED",0,0,0

    n_modalities=z["output_type"].nunique()
    track_cols=[c for c in ["track_name","biosample_name","ontology_curie"] if c in z.columns]
    if track_cols:
        n_tracks=z[track_cols].astype(str).drop_duplicates().shape[0]
    else:
        n_tracks=len(z)

    if n_modalities>=2 or n_tracks>=2:
        return "RNA_LEVEL_POSITIVE",len(z),n_modalities,n_tracks
    return "RNA_LEVEL_SINGLE_SIGNAL_HOLD",len(z),n_modalities,n_tracks

summary=[]
for name in VARIANTS:
    sub=bub_primary[bub_primary.variant_name==name]
    call,n_extreme,n_mod,n_tracks=classify_variant(sub)
    max_q=float(sub.abs_quantile.max()) if len(sub) else np.nan
    summary.append({
        "variant_name":name,
        "mechanism_call":call,
        "n_extreme_primary_rows":n_extreme,
        "n_extreme_primary_modalities":n_mod,
        "n_extreme_primary_tracks":n_tracks,
        "max_abs_primary_quantile":max_q,
    })

summary=pd.DataFrame(summary)
display(summary)
summary.to_csv(
    PUBLIC_OUT/"NB04C_ALPHAGENOME_MECHANISM_SUMMARY.csv",
    index=False,
)

## 5. Secondary discovery audit

In [ ]:
secondary=scores_df[scores_df["output_type"].isin(SECONDARY)].copy()
secondary["abs_quantile"]=secondary["quantile_score"].abs()
secondary_extreme=secondary[secondary.abs_quantile>=0.995].copy()

# Secondary outputs are descriptive only: they cannot change the primary RNA-level call.
sec_cols=[
    c for c in [
        "variant_name","output_type","biosample_name","gtex_tissue",
        "ontology_curie","track_name","raw_score","quantile_score","abs_quantile"
    ] if c in secondary_extreme.columns
]

secondary_extreme[sec_cols].sort_values(
    ["variant_name","abs_quantile"],ascending=[True,False]
).to_csv(
    PUBLIC_OUT/"NB04C_ALPHAGENOME_SECONDARY_EXTREME.csv",
    index=False,
)

print("Secondary extreme rows:",len(secondary_extreme))
display(secondary_extreme[sec_cols].head(80))

## 6. Sanitized handoff to Track 2 narrative

In [ ]:
from datetime import datetime, timezone

calls=dict(zip(summary.variant_name,summary.mechanism_call))

if "RNA_LEVEL_POSITIVE" in calls.values():
    experiment_update=(
        "Add targeted RNA validation (RT-PCR/junction assay and, if feasible, RNA-seq) "
        "before interpreting BUBR1 protein abundance or drug-rescue experiments."
    )
else:
    experiment_update=(
        "No strong AlphaGenome RNA-level mechanism was detected under the frozen rule. "
        "Proceed with protein abundance/half-life/SAC as the primary unresolved mechanism."
    )

handoff={
    "schema":"MVA-NB04C-ALPHAGENOME-RNA-MECHANISM-1.0",
    "created_utc":datetime.now(timezone.utc).isoformat(),
    "status":"COMPLETE",
    "purpose":"orthogonal DNA-to-RNA mechanism audit; not protein-stability prediction",
    "variants":{
        "L737star":"chr15:40209701:T>G",
        "N1002K":"chr15:40220612:T>G",
    },
    "primary_modalities":PRIMARY,
    "extreme_threshold_abs_quantile":0.995,
    "calls":calls,
    "experiment_update":experiment_update,
    "acmg_rule":{
        "alphagenome_direct_PS3":False,
        "stack_as_independent_PP3_vote":False,
        "use":"mechanism triage and experimental design only unless separately validated under ACMG/ClinGen rules"
    },
    "privacy":{
        "private_VCF_used":False,
        "private_genotypes_used":False,
        "only_two_already_public_BUB1B_variants_sent_to_API":True
    },
    "limitations":[
        "AlphaGenome predicts molecular consequences from reference-sequence context, not personal-genome effects.",
        "It does not directly predict BUBR1 protein folding, stability, half-life, or SAC function.",
        "Tissue/cell-state specificity is imperfect.",
        "Predictions require orthogonal experimental confirmation."
    ]
}

(DOC_OUT/"NB04C_ALPHAGENOME_HANDOFF.json").write_text(
    json.dumps(handoff,indent=2)
)

print(json.dumps(handoff,indent=2))